# Eau et agriculture : qualité des eaux et usage des pesticides

**Problématique :** _à formuler_

Auteurs : _à compléter_

## 0. Imports et configuration

In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from src.hubeau.construire_eau_zones import construire_eau_zones
from src.geo.recuperer_departements import recuperer_departements
from src.geo.recuperer_zones_hydro import recuperer_zones_hydro
from src.agriculture.recuperer_rpg_ign import recuperer_rpg_ign
from src.agregation.parcelles_pesticides.constantes_agregation import LISTE_COLONNES_MOLECULES
from src.nettoyage.agreger_champs_zones import agreger_champs_zones
from src.nettoyage.agreger_pesticides_zones import agreger_pesticides_zones
from src.nettoyage.associer_parcelles_zones import associer_parcelles_zones
from src.nettoyage.construire_base_zones import construire_base_zones
from src.telecharger_donnee import telecharger_donnee
from src.visualisation.tracer_evolution_indices import tracer_evolution_indices
from src.visualisation.tracer_carte import tracer_carte

plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e5e5e5",
})

## 1. Récupération des données

1.1 Stations hydrobiologiques (Hub'Eau)  
1.2 Indices biologiques de qualité des cours d'eau (Hub'Eau)  
1.3 Achats de produits phytosanitaires par département  
1.4 Données agricoles (RPG)  
1.5 Fonds de carte  
1.6 Zones hydrographiques

Chaque fichier passe par `telecharger_donnee` : il est pris en local s'il existe, sinon dans la release [`donnees-v1`](https://github.com/raphaelcoic/Projet_python_data_science/releases/tag/donnees-v1). S'il n'y est pas, il est créé puis déposé dans la release. Pour régénérer un fichier, on passe `recalculer=True` : il est recréé et remplace sa copie dans la release.

Pour la partie eau, seule la table finale par zone hydrographique (`eau_zones.parquet`, partie 2.2) est dans la release. Les stations et les indices ne sont demandés à l'API Hub'Eau, ce qui prend plusieurs minutes, que si `eau_zones.parquet` manque ou qu'on le recalcule.

### 1.1 Stations hydrobiologiques

Coordonnées des stations où sont calculés les indices biologiques, dans les 94 départements de métropole continentale. La Corse est exclue de tout le projet : le RPG y est trop incomplet.

### 1.2 Indices biologiques

La qualité d'une rivière se mesure aussi par les êtres vivants qu'on y trouve. Trois indices, calculés sur 2013-2023 :

| Indice | Ce qu'il observe | Échelle | Sensible à |
|---|---|---|---|
| **I2M2** | invertébrés (larves, vers, mollusques) | 0 à 1, **haut = bonne qualité** | pollutions multiples, dont les pesticides |
| **IBD** | diatomées (micro-algues) | 0 à 20, **haut = bonne qualité** | nutriments (engrais) et matière organique |
| **IPR** | poissons | 0 et plus, **bas = bonne qualité** | état physique de la rivière, pollutions |

### 1.5 Fonds de carte

Contours simplifiés des départements ([france-geojson](https://github.com/gregoiredavid/france-geojson)), projetés en Lambert 93 (EPSG:2154), la projection officielle de la métropole.

In [ ]:
departements = gpd.read_parquet(telecharger_donnee("data/geo/departements.parquet", recuperer_departements))
departements = departements.to_crs("EPSG:2154")

### 1.6 Zones hydrographiques

Les 6 190 zones hydrographiques de la [BD TOPAGE](https://www.sandre.eaufrance.fr/atlas/srv/fre/catalog.search#/home) découpent la France en petits bassins versants. On retire les 80 zones de Corse.

In [ ]:
zones = gpd.read_parquet(telecharger_donnee("data/geo/zones_hydro.parquet", recuperer_zones_hydro))
zones = zones[~zones["code_zone"].str.startswith("12")]  # zones de Corse
zones.shape

## 2. Nettoyage et mise en forme

### 2.1 Indices biologiques

On retire :
- les valeurs que le producteur ne qualifie pas de « correctes » (environ 9 % des lignes) ;
- les valeurs manquantes et les valeurs codées 999 de l'IPR (indice non calculable) ;
- les doublons exacts.

Quand une station a plusieurs valeurs d'un même indice le même jour (environ 4 400 cas), on garde leur moyenne : on a ainsi une seule valeur par station, date et indice.

Après ce nettoyage, il ne reste aucune valeur manquante et toutes les valeurs sont dans l'échelle de leur indice (0 à 1 pour l'I2M2, 0 à 20 pour l'IBD). L'IPR n'a pas de borne haute : ses valeurs élevées (jusqu'à 165) signalent des rivières très dégradées, on les garde.

### 2.2 Indices par zone hydrographique

On travaille aussi à la maille de la **zone hydrographique**, plus fine que le département et qui suit l'écoulement de l'eau. Chaque station ayant un indice va dans la zone qui la contient, ou dans la plus proche si elle est juste hors limites. Une zone ne couvre que le bassin local : pour une station sur un grand cours d'eau, une partie de l'amont reste en dehors.

La table finale de la partie eau, `eau_zones.parquet`, agrège les indices à la zone **pour chaque année** : pour chaque indice, la moyenne des mesures de chaque station dans l'année, puis la médiane des stations de la zone (moins sensible que la moyenne aux valeurs extrêmes de l'IPR).

In [ ]:
eau_zones = pd.read_parquet(telecharger_donnee(
    "data/processed/eau_zones.parquet",
    lambda: construire_eau_zones(zones),
))

print("doublons sur la clé :", eau_zones.duplicated(subset=["code_zone", "annee"]).sum())
eau_zones.shape

## 3. Statistiques descriptives

### 3.1 Distribution des indices (2013-2023)

Valeurs de chaque indice par zone et par année (`eau_zones`, partie 2.2).

In [ ]:
eau_zones[["I2M2", "IBD", "IPR"]].describe().round(2)

Les trois indices sont très dispersés d'une zone à l'autre : l'I2M2 couvre toute l'échelle de 0 à 1. L'IPR a quelques valeurs extrêmes (jusqu'à 165) : on s'appuie donc sur la **médiane** plutôt que sur la moyenne.

### 3.2 Évolution nationale 2013-2023

Pour chaque année, médiane des zones suivies cette année-là (`eau_zones`). L'axe de l'IPR est inversé : sur les trois panneaux, **le haut signifie une meilleure qualité**.

In [ ]:
tracer_evolution_indices(eau_zones);

Les variations sont faibles (attention, les axes ne partent pas de 0) : la médiane de l'I2M2 reste entre 0,53 et 0,61, celle de l'IBD entre 15,1 et 15,5. L'I2M2 baisse depuis 2017 ; l'IPR se dégrade, avec un saut en 2017 (de 15,4 en 2013 à 16,4 en 2023) ; l'IBD est stable. Les zones suivies ne sont pas toutes les mêmes d'une année à l'autre, ce qui peut suffire à créer des écarts de cette taille, comme le saut de 2017.

## 4. Visualisations et cartes

### 4.1 Qualité biologique par zone hydrographique

Valeur de chaque indice par zone (`eau_zones`, partie 2.2), pour une année : 2023 par défaut, l'année du RPG utilisé en partie 5. Pour voir une autre année, il suffit de changer `ANNEE_CARTE`. La palette de l'IPR est inversée : sur les trois cartes, **foncé = meilleure qualité**. Chaque carte a sa propre échelle, du 2e au 98e centile. Les zones sans station suivie cette année-là sont en gris.

In [ ]:
ANNEE_CARTE = 2023
eau_annee = eau_zones[eau_zones["annee"] == ANNEE_CARTE]

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
tracer_carte(zones, eau_annee, "I2M2", f"I2M2 (invertébrés), {ANNEE_CARTE}", axes[0])
tracer_carte(zones, eau_annee, "IBD", f"IBD (diatomées), {ANNEE_CARTE}", axes[1])
tracer_carte(zones, eau_annee, "IPR", f"IPR (poissons), {ANNEE_CARTE}", axes[2], cmap="Blues_r")

Pour l'I2M2 et l'IBD, la qualité est meilleure dans les massifs (Massif central, Alpes, Pyrénées) et plus faible dans le Bassin parisien et le Centre-Ouest. Mais la qualité varie fortement **à l'intérieur** d'un même département, d'une zone à sa voisine : c'est ce qui justifie le croisement avec les cultures à cette maille (partie 5).

Limites, pour 2023 :
- environ 3 400 des 6 110 zones ont au moins une station suivie cette année-là, et l'IPR, moins suivi, n'est disponible que pour environ 1 400 zones : sa carte est difficile à lire ;
- dans deux zones suivies sur trois, la valeur de l'année repose sur une seule station.

### 4.2 Évolution 2013-2023 par zone hydrographique

Taux d'évolution de chaque indice entre la première et la dernière année, en %, pour les zones suivies ces deux années-là. Les palettes sont orientées pour que, sur les trois cartes, **bleu = amélioration** et **rouge = dégradation** : pour l'IPR, une hausse est une dégradation.

In [ ]:
ANNEE_DEBUT, ANNEE_FIN = 2013, 2023
debut = eau_zones[eau_zones["annee"] == ANNEE_DEBUT].set_index("code_zone")[["I2M2", "IBD", "IPR"]]
fin = eau_zones[eau_zones["annee"] == ANNEE_FIN].set_index("code_zone")[["I2M2", "IBD", "IPR"]]
evolution = ((fin - debut) / debut.where(debut > 0) * 100).reset_index()  # pas de taux si la valeur de départ est nulle
print("zones avec un taux :", evolution[["I2M2", "IBD", "IPR"]].count().to_dict())

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
tracer_carte(zones, evolution, "I2M2", f"I2M2, évolution {ANNEE_DEBUT}-{ANNEE_FIN} (%)", axes[0], cmap="RdBu", centrer=True)
tracer_carte(zones, evolution, "IBD", f"IBD, évolution {ANNEE_DEBUT}-{ANNEE_FIN} (%)", axes[1], cmap="RdBu", centrer=True)
tracer_carte(zones, evolution, "IPR", f"IPR, évolution {ANNEE_DEBUT}-{ANNEE_FIN} (%)", axes[2], cmap="RdBu_r", centrer=True)

Il n'y a pas de tendance d'ensemble : entre 2013 et 2023, l'I2M2 progresse dans environ la moitié des zones, l'IBD dans 46 % d'entre elles, et l'IPR s'améliore dans 40 % seulement. Aucune région ne se détache nettement, les zones qui s'améliorent et celles qui se dégradent sont souvent voisines.

Ces cartes sont à lire avec prudence :
- seules les zones suivies en 2013 **et** en 2023 ont un taux (environ 1 400 pour l'I2M2, 2 200 pour l'IBD, 1 000 pour l'IPR). Pour l'I2M2, l'ouest de la France manque presque entièrement : le réseau n'a été étendu qu'en 2015 ;
- chaque taux compare deux années isolées, souvent une seule mesure par zone : une partie des écarts est du bruit d'une année sur l'autre.

## 5. Croisement eau × agriculture

On croise l'eau et les cultures à la maille de la **zone hydrographique** : les parcelles sont rattachées aux zones comme les stations (partie 2.2).

### 5.1 Parcelles et zones

Les parcelles viennent de l'archive nationale du RPG 2023 publiée par l'IGN, qui donne le contour de chaque parcelle.

Une parcelle qui touche une seule zone y va entière. Une parcelle qui en touche plusieurs est coupée selon leurs limites, et sa surface est répartie au prorata de chaque morceau. Si `champs_zones.parquet` n'est pas dans la release, l'archive (2,75 Go) est téléchargée et le calcul prend environ une demi-heure.

In [ ]:
champs_zones = pd.read_parquet(telecharger_donnee(
    "data/processed/champs_zones.parquet",
    lambda: agreger_champs_zones(recuperer_rpg_ign(), zones),
))

print("doublons sur la clé :", champs_zones.duplicated(subset=["code_zone", "code_groupe_culture"]).sum())
print("surface rattachée (millions d'ha) :", round(champs_zones["surface_ha"].sum() / 1e6, 2))

### 5.2 Pesticides par zone

Tomia estime la quantité de chacune des 10 molécules attribuée à chaque parcelle, pour chaque année (`src/agregation/parcelles_pesticides/`) : les achats du département sont répartis entre les parcelles des groupes de cultures qui utilisent la molécule, au prorata de leur surface. On les ramène à la zone hydrographique en deux étapes.

1. **Chaque parcelle et sa ou ses zones** (`associer_parcelles_zones`) : une parcelle qui touche une seule zone y va entière (`part` = 1). Une parcelle qui en touche plusieurs est coupée par leurs limites, et chaque zone reçoit la part de la parcelle qui tombe chez elle ; les parts d'une parcelle font 1.

In [ ]:
parcelles_zones = pd.read_parquet(telecharger_donnee(
    "data/processed/parcelles_zones.parquet",
    lambda: associer_parcelles_zones(recuperer_rpg_ign(), zones),
))

print("parcelles :", parcelles_zones["id_parcelle"].nunique())
print("parcelles partagées entre plusieurs zones :", parcelles_zones.loc[parcelles_zones["part"] < 1, "id_parcelle"].nunique())

2. **Somme par zone** (`agreger_pesticides_zones`) : pour chaque année, les quantités de chaque parcelle sont multipliées par sa part dans la zone, puis additionnées par zone. Les fichiers par année de Tomia (`data/processed/parcelles_pesticides/`) se génèrent avec `python -m src.agregation.parcelles_pesticides.construire_parcelles_pesticides`.

In [ ]:
fichiers = sorted(Path("data/processed/parcelles_pesticides").glob("parcelles_pesticides_*.parquet"))
pesticides_zones = pd.read_parquet(telecharger_donnee(
    "data/processed/pesticides_zones.parquet",
    lambda: pd.concat([
        agreger_pesticides_zones(pd.read_parquet(f, columns=["id_parcelle", "annee", *LISTE_COLONNES_MOLECULES]), parcelles_zones)
        for f in fichiers
    ]),
))

print("doublons sur la clé :", pesticides_zones.duplicated(subset=["code_zone", "annee"]).sum())
pesticides_zones.groupby("annee")[LISTE_COLONNES_MOLECULES].sum().round(0)

### 5.3 Table croisée : `base_zones.parquet`

Une ligne par zone suivie et par année : la médiane de chaque indice la part de la zone occupée par chaque groupe de cultures et les pesticides attribués à la zone cette année-là, en kg par hectare de zone. Les cultures sont celles du RPG 2023, reprises pour toutes les années.

In [ ]:
base_zones = pd.read_parquet(telecharger_donnee(
    "data/processed/base_zones.parquet",
    lambda: construire_base_zones(eau_zones, champs_zones, pesticides_zones, zones),
))

print("doublons sur la clé :", base_zones.duplicated(subset=["code_zone", "annee"]).sum())
base_zones.shape

## 6. Conclusion et limites